In [1]:
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

In [2]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [11]:
play_store_df = pd.read_csv(r"C:\Users\ADMIN\OneDrive\Desktop\elevance skills project\Play Store Data.csv")
play_store_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [9]:
reviews_df = pd.read_csv(r"C:\Users\ADMIN\OneDrive\Desktop\elevance skills project\User Reviews.csv")
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [13]:
#data cleaning
play_store_df = play_store_df.dropna(subset=['Rating'])
for column in play_store_df.columns:
    play_store_df[column].fillna(play_store_df[column].mode()[0], inplace=True)

play_store_df.drop_duplicates(inplace=True)

play_store_df = play_store_df[play_store_df['Rating'] <= 5]


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17764\2534183683.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  play_store_df[column].fillna(play_store_df[column].mode()[0], inplace=True)


In [14]:
reviews_df.dropna(subset=['Translated_Review'], inplace=True)

In [15]:
#converts installs column
play_store_df["Installs_Num"] = (
    play_store_df["Installs"]
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

play_store_df["Installs_Num"] = pd.to_numeric(
    play_store_df["Installs_Num"],
    errors="coerce"
)

In [17]:
play_store_df[["Installs", "Installs_Num"]].head()

,Installs,Installs_Num
0,"10,000+",10000
1,"500,000+",500000
2,"5,000,000+",5000000
3,"50,000,000+",50000000
4,"100,000+",100000


In [18]:
#mergingboth csv files
merged_df = pd.merge(
    play_store_df,
    reviews_df,
    on="App",
    how="inner"
)

In [19]:
merged_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Installs_Num,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000,A kid's excessive ads. The types ads allowed a...,Negative,-0.250,1.000000
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000,It bad >:(,Negative,-0.725,0.833333
2,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000,like,Neutral,0.000,0.000000
3,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000,I love colors inspyering,Positive,0.500,0.600000
4,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000,I hate,Negative,-0.800,0.900000


In [20]:
merged_df.columns

Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver', 'Installs_Num', 'Translated_Review', 'Sentiment',
       'Sentiment_Polarity', 'Sentiment_Subjectivity'],
      dtype='str')

In [21]:
#filtering the data
filtered_df = merged_df[
    merged_df["Category"].str.startswith(("B", "C", "E"))
]
filtered_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Installs_Num,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
671,"BestCam Selfie-selfie, beauty camera, photo ed...",BEAUTY,3.9,1739,21M,"500,000+",Free,0,Everyone,Beauty,"July 12, 2018",1.0.6,4.0.3 and up,500000,This a̸p̸p̸ Na Kare please Mere Bhai A L pleas...,Negative,-0.500000,0.950000
672,"BestCam Selfie-selfie, beauty camera, photo ed...",BEAUTY,3.9,1739,21M,"500,000+",Free,0,Everyone,Beauty,"July 12, 2018",1.0.6,4.0.3 and up,500000,Worst lot add; time west,Negative,-1.000000,1.000000
673,"BestCam Selfie-selfie, beauty camera, photo ed...",BEAUTY,3.9,1739,21M,"500,000+",Free,0,Everyone,Beauty,"July 12, 2018",1.0.6,4.0.3 and up,500000,bed bakvas time west stupid,Negative,-0.800000,1.000000
674,"BestCam Selfie-selfie, beauty camera, photo ed...",BEAUTY,3.9,1739,21M,"500,000+",Free,0,Everyone,Beauty,"July 12, 2018",1.0.6,4.0.3 and up,500000,It bad dont install worst,Negative,-0.850000,0.833333
675,"BestCam Selfie-selfie, beauty camera, photo ed...",BEAUTY,3.9,1739,21M,"500,000+",Free,0,Everyone,Beauty,"July 12, 2018",1.0.6,4.0.3 and up,500000,Fake install best sweet selfie,Positive,0.283333,0.650000


In [22]:
filtered_df["Category"].unique()

<ArrowStringArray>
[             'BEAUTY', 'BOOKS_AND_REFERENCE',            'BUSINESS',
              'COMICS',       'COMMUNICATION',           'EDUCATION',
       'ENTERTAINMENT',              'EVENTS']
Length: 8, dtype: str

In [23]:
filtered_df["Reviews"] = pd.to_numeric(
    filtered_df["Reviews"],
    errors="coerce"
)

In [25]:
filtered_df = filtered_df[
    filtered_df["Reviews"] > 500
]
filtered_df["Reviews"].head()

671    1739
672    1739
673    1739
674    1739
675    1739
Name: Reviews, dtype: int64

In [29]:
filtered_df = filtered_df[
    ~filtered_df["App"].str.startswith(("X", "Y", "Z"))
]

In [27]:
filtered_df = filtered_df[
    ~filtered_df["App"].str.contains("S", case=False, na=False)
]

In [32]:
translation_dict = {
    "BEAUTY": "सौंदर्य",
    "BUSINESS": "வணிகம்",
    "DATING": "Partnersuche"
}

filtered_df["Display_Category"] = (
    filtered_df["Category"].replace(translation_dict)
)

In [37]:
filtered_df[["Category", "Display_Category"]].head(100)

,Category,Display_Category
1009,BOOKS_AND_REFERENCE,BOOKS_AND_REFERENCE
1010,BOOKS_AND_REFERENCE,BOOKS_AND_REFERENCE
1011,BOOKS_AND_REFERENCE,BOOKS_AND_REFERENCE
1012,BOOKS_AND_REFERENCE,BOOKS_AND_REFERENCE
1013,BOOKS_AND_REFERENCE,BOOKS_AND_REFERENCE
...,...,...
1104,BOOKS_AND_REFERENCE,BOOKS_AND_REFERENCE
1105,BOOKS_AND_REFERENCE,BOOKS_AND_REFERENCE
1106,BOOKS_AND_REFERENCE,BOOKS_AND_REFERENCE
1107,BOOKS_AND_REFERENCE,BOOKS_AND_REFERENCE


In [36]:
filtered_df.columns

Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver', 'Installs_Num', 'Translated_Review', 'Sentiment',
       'Sentiment_Polarity', 'Sentiment_Subjectivity', 'Display_Category'],
      dtype='str')

In [38]:
filtered_df[
    filtered_df["Category"].isin(["BEAUTY", "BUSINESS", "DATING"])
][["Category", "Display_Category"]]

,Category,Display_Category
1790,BUSINESS,வணிகம்
1791,BUSINESS,வணிகம்
1826,BUSINESS,வணிகம்
1827,BUSINESS,வணிகம்
1828,BUSINESS,வணிகம்
...,...,...
1941,BUSINESS,வணிகம்
1942,BUSINESS,வணிகம்
1943,BUSINESS,வணிகம்
1944,BUSINESS,வணிகம்


In [40]:
filtered_df["Category"].dtype

<StringDtype(na_value=nan)>

In [41]:
merged_df["Category"] = merged_df["Category"].astype(str)

In [ ]:
filtered_df = merged_df[
    merged_df["Category"].str.startswith(("B", "C", "E"), na=False)
]
